## Step 4 : Player curves and the draft top 10s

Reads `outputs/balls_clean.parquet`, `outputs/balls_pressure.parquet` and `outputs/balls_value.parquet` (steps 1 to 3).
Writes `outputs/04_batters.csv` and `outputs/04_bowlers.csv` (every eligible player, ranked).

For every player I measure two things:
1. **Level** : value per ball over all his balls (runs above an average player in the same situation, wickets counted as runs)
2. **Clutch** : how much better or worse he is on high pressure balls than on his other balls

Both are shrunk towards the average. The fewer balls a player has, the harder the pull, so one lucky over cannot put anyone at rank 1.
The score is the value I expect from him on a high pressure ball = level + clutch. Clutch is only used if it passes a noise test.

In [9]:
# Importing Libraries
import os
import numpy as np
import pandas as pd

output_folder = os.path.join("..", "outputs")
keys = ['match_id', 'innings', 'innings_ball_num']

# Loading Dataset (the 3 tables from steps 1 to 3, one row per ball)
balls = pd.read_parquet(os.path.join(output_folder, "balls_clean.parquet"), columns=keys + [
    'striker_id', 'striker_name', 'bowler_id', 'bowler_name', 'batting_team_id', 'bowling_team_id',
    'legal', 'comp_group', 'pace_or_spin'])
balls = balls.merge(pd.read_parquet(os.path.join(output_folder, "balls_pressure.parquet"), columns=keys + ['pressure']), on=keys)
balls = balls.merge(pd.read_parquet(os.path.join(output_folder, "balls_value.parquet"), columns=keys + ['batter_value', 'bowler_value']), on=keys)

# The 30 Other T20 matches were only used to train the models, their players are not ranked
balls = balls[balls['comp_group'] != 'Other T20']
print(balls.shape)

(193988, 15)


In [10]:
# Does pressure change the value of an average player? (10 groups of pressure, both should stay near 0)
# If it did, every player's clutch number would be shifted by the same amount
balls['pressure_group'] = (balls['pressure'] // 10).clip(upper=9).astype(int)
print(balls.groupby('pressure_group').agg(
    batters=('batter_value', 'mean'), bowlers=('bowler_value', 'mean'), balls=('pressure', 'size')).round(3).T)

# High pressure ball = pressure 80 or more, the top fifth of all balls
balls['high'] = balls['pressure'] >= 80
print("share of high pressure balls:", round(balls['high'].mean(), 3))

pressure_group          0          1          2          3          4          5          6          7          8          9
batters             0.032      0.017     -0.008     -0.016     -0.015      0.019      0.009     -0.027      0.011     -0.001
bowlers            -0.016     -0.017      0.006      0.015      0.012     -0.021     -0.013      0.023     -0.011     -0.003
balls           19144.000  19512.000  19515.000  19533.000  19539.000  19487.000  19378.000  19391.000  19410.000  19079.000
share of high pressure balls: 0.198


In [11]:
# Level and clutch of every player, shrunk towards the average
#
# level  = his average value per ball over all balls
# delta  = his average on high pressure balls minus his average on the other balls
# Shrinking : a raw average = true skill + luck. If the true skill varies across players by tau^2 and the luck in
# his average is L, the best guess of his skill is average + (raw - average) * tau^2 / (tau^2 + L).
# The level is pulled towards the average level of the pool (players with enough high pressure balls), the delta towards 0.
# tau^2 is measured from the pool : spread of the raw numbers minus the average luck.
# If tau^2 comes out as 0, the spread is all luck and every player's number is pulled all the way.
#
# Luck : spread of a single ball / number of balls, times one factor for the fact that balls in the same match
# share the pitch and conditions and are not independent. The factor = luck measured match by match / luck measured
# ball by ball, averaged over the players in the pool. One factor for everyone, because a player's own
# match by match luck from 10 to 30 matches is too noisy and reshuffled the list by itself.
# For the delta the match by match luck is taken on the difference itself : a good pitch lifts his high pressure
# balls AND his other balls of that match, so part of the luck cancels in the difference.
#
# Value on a high pressure ball = level + share of his balls that are NOT high pressure * delta
# (because his overall average already has the high pressure balls in it)
def player_curves(balls, player, name, team, value, min_high, show=True):
    per_player = balls.groupby(player)
    table = pd.DataFrame({
        'name': per_player[name].first(),
        'team': per_player[team].agg(lambda x: x.mode()[0]),
        'balls': per_player.size(),
        'high_balls': balls[balls['high']].groupby(player).size(),
        'matches': per_player['match_id'].nunique(),
        'first_innings_share': per_player['innings'].agg(lambda x: (x == 1).mean()),
        'raw_level': per_player[value].mean(),
        'raw_high': balls[balls['high']].groupby(player)[value].mean(),
        'raw_other': balls[~balls['high']].groupby(player)[value].mean(),
    }).fillna({'high_balls': 0})
    table['delta'] = table['raw_high'] - table['raw_other']
    other_balls = table['balls'] - table['high_balls']
    in_pool = table['high_balls'] >= min_high

    # Ball by ball luck
    high = balls['high'].values
    gap_level = balls[value] - per_player[value].transform('mean')
    gap_split = balls[value] - balls.groupby([player, 'high'])[value].transform('mean')
    luck_level = gap_level.var() / table['balls']
    luck_delta = gap_split[high].var() / table['high_balls'] + gap_split[~high].var() / other_balls

    # Match by match luck = sum over his matches of (what the match adds to his number)^2
    match_keys = [balls[player], balls['match_id']]
    in_level = gap_level.groupby(match_keys).sum() / table['balls'].reindex(balls[player].unique()).rename_axis(player)
    weight = pd.Series(np.where(high, 1 / balls[player].map(table['high_balls']), -1 / balls[player].map(other_balls)), index=balls.index)
    in_delta = (gap_split * weight).groupby(match_keys).sum()
    factor_level = (in_level ** 2).groupby(level=0).sum()[in_pool].mean() / luck_level[in_pool].mean()
    factor_delta = (in_delta ** 2).groupby(level=0).sum()[in_pool].mean() / luck_delta[in_pool].mean()
    table['luck_level'] = luck_level * factor_level
    table['luck_delta'] = luck_delta * factor_delta

    pool = table[in_pool]
    tau_level = max(pool['raw_level'].var() - pool['luck_level'].mean(), 0)
    tau_delta = max(pool['delta'].var() - pool['luck_delta'].mean(), 0)

    pool_level = pool['raw_level'].mean()
    table['level'] = pool_level + (table['raw_level'] - pool_level) * tau_level / (tau_level + table['luck_level'])
    table['clutch'] = (table['delta'] * tau_delta / (tau_delta + table['luck_delta'])).fillna(0) \
        * (1 - table['high_balls'] / table['balls'])
    table['score'] = table['level'] + table['clutch']
    table.attrs['clutch_spread'] = np.sqrt(tau_delta)

    # Consistency : in the matches where he had 6 or more high pressure balls, how often was he above the average player?
    per_match = balls[balls['high']].groupby([player, 'match_id'])[value].agg(['sum', 'size'])
    per_match = per_match[per_match['size'] >= 6]
    table['pressure_matches'] = per_match.groupby(player).size()
    table['beat_average'] = (per_match['sum'] > 0).groupby(player).mean()
    table = table.fillna({'pressure_matches': 0})

    if show:
        print(f"{value}: true spread of level {np.sqrt(tau_level):.3f}, true spread of clutch {np.sqrt(tau_delta):.3f}, "
              f"raw spread of clutch {pool['delta'].std():.3f}, luck in it {np.sqrt(pool['luck_delta'].mean()):.3f}, "
              f"match factor {factor_level:.2f} / {factor_delta:.2f}, pool average level {pool_level:+.3f} "
              f"(runs per ball, {len(pool)} players with {min_high}+ high pressure balls)")
    return table

In [12]:
# Batters on legal balls (a wide is not a ball faced), bowlers on every delivery
batters = player_curves(balls[balls['legal']], 'striker_id', 'striker_name', 'batting_team_id', 'batter_value', min_high=100)
bowlers = player_curves(balls, 'bowler_id', 'bowler_name', 'bowling_team_id', 'bowler_value', min_high=120)

# Does the result depend on where high pressure starts? Same numbers with the cut at 70 and at 90
for cut in [70, 90]:
    test = balls.assign(high=balls['pressure'] >= cut)
    print(f"cut at {cut}:")
    player_curves(test[test['legal']], 'striker_id', 'striker_name', 'batting_team_id', 'batter_value', min_high=100)
    player_curves(test, 'bowler_id', 'bowler_name', 'bowling_team_id', 'bowler_value', min_high=120)

batter_value: true spread of level 0.103, true spread of clutch 0.000, raw spread of clutch 0.216, luck in it 0.227, match factor 1.12 / 1.00, pool average level +0.057 (runs per ball, 121 players with 100+ high pressure balls)
bowler_value: true spread of level 0.072, true spread of clutch 0.057, raw spread of clutch 0.211, luck in it 0.203, match factor 1.10 / 1.02, pool average level +0.021 (runs per ball, 102 players with 120+ high pressure balls)
cut at 70:
batter_value: true spread of level 0.109, true spread of clutch 0.000, raw spread of clutch 0.230, luck in it 0.238, match factor 1.20 / 1.06, pool average level +0.024 (runs per ball, 162 players with 100+ high pressure balls)
bowler_value: true spread of level 0.079, true spread of clutch 0.037, raw spread of clutch 0.221, luck in it 0.217, match factor 1.14 / 1.05, pool average level -0.001 (runs per ball, 153 players with 120+ high pressure balls)
cut at 90:
batter_value: true spread of level 0.076, true spread of clutch 0.

In [13]:
# Is the clutch spread real, and could I even see one? Two tests of the method itself
#
# Test 1 (no clutch) : inside every player I shuffle which of his balls are called high pressure. Now nobody can
# have a clutch skill, so whatever spread the method finds is noise. I only use clutch if the real spread is
# above 90% of the shuffles.
# Test 2 (known clutch) : I add a made up clutch skill (spread 0.10 runs per ball) to the high pressure balls
# and look at what the method finds. It shows how small a real clutch skill could hide in this data.
shuffler = np.random.RandomState(7)
groups = {
    'batters': (balls[balls['legal']], 'striker_id', 'striker_name', 'batting_team_id', 'batter_value', 100, batters),
    'bowlers': (balls, 'bowler_id', 'bowler_name', 'bowling_team_id', 'bowler_value', 120, bowlers),
}
for label, (source, player, name, team, value, min_high, table) in groups.items():
    shuffled, added = [], []
    for run in range(30):
        test = source.copy()
        test['high'] = test.groupby(player)['high'].transform(lambda x: shuffler.permutation(x.values))
        shuffled.append(player_curves(test, player, name, team, value, min_high, show=False).attrs['clutch_spread'])
    for run in range(10):
        test = source.copy()
        ids = test[player].unique()
        skill = pd.Series(shuffler.normal(0, 0.10, len(ids)), index=ids)
        test[value] = test[value] + test['high'] * test[player].map(skill).values
        added.append(player_curves(test, player, name, team, value, min_high, show=False).attrs['clutch_spread'])
    real = table.attrs['clutch_spread']
    limit = np.percentile(shuffled, 90)
    print(f"{label}: real clutch spread {real:.3f}. Shuffled: average {np.mean(shuffled):.3f}, 90% are below {limit:.3f}, "
          f"{np.mean(np.array(shuffled) >= real):.0%} of shuffles are as big as the real one.")
    print(f"   with a made up clutch of 0.10 added the method finds {np.mean(added):.3f} on average ({min(added):.3f} to {max(added):.3f})")
    if real <= limit:
        table['clutch'] = 0.0
        table['score'] = table['level']
        print("   -> not above the noise, clutch is set to 0 and the score is the level")

batters: real clutch spread 0.000. Shuffled: average 0.054, 90% are below 0.103, 100% of shuffles are as big as the real one.
   with a made up clutch of 0.10 added the method finds 0.053 on average (0.000 to 0.106)
   -> not above the noise, clutch is set to 0 and the score is the level
bowlers: real clutch spread 0.057. Shuffled: average 0.051, 90% are below 0.099, 50% of shuffles are as big as the real one.
   with a made up clutch of 0.10 added the method finds 0.104 on average (0.056 to 0.149)
   -> not above the noise, clutch is set to 0 and the score is the level


In [14]:
# Eligibility : how many players are left for a few minimums of high pressure balls and pressure matches
for label, table, minimums in [('batters', batters, [60, 100, 150]), ('bowlers', bowlers, [60, 120, 180])]:
    for min_high in minimums:
        counts = [int(((table['high_balls'] >= min_high) & (table['pressure_matches'] >= m)).sum()) for m in [5, 10, 15]]
        print(f"{label}: {min_high}+ high pressure balls and 5 / 10 / 15+ pressure matches -> {counts} players")

batters: 60+ high pressure balls and 5 / 10 / 15+ pressure matches -> [150, 67, 30] players
batters: 100+ high pressure balls and 5 / 10 / 15+ pressure matches -> [120, 67, 30] players
batters: 150+ high pressure balls and 5 / 10 / 15+ pressure matches -> [79, 64, 30] players
bowlers: 60+ high pressure balls and 5 / 10 / 15+ pressure matches -> [153, 85, 53] players
bowlers: 120+ high pressure balls and 5 / 10 / 15+ pressure matches -> [102, 85, 53] players
bowlers: 180+ high pressure balls and 5 / 10 / 15+ pressure matches -> [69, 69, 53] players


In [15]:
# My pick : 100+ high pressure balls (batters) or 120+ (bowlers, they bowl more balls in a match),
# and 10+ pressure matches (a match with 6 or more high pressure balls), about one full season of tight finishes.
# Raising the ball minimum barely changes the list, the match minimum is the one that bites.
#
# IPL rule : the report is for the Rajasthan Royals and the IPL 2027 auction, so a player must also have a real
# IPL sample : half of the ball minimum and 5+ pressure matches in the IPL itself. His T20I balls still count as
# evidence, but a player with no IPL record (for example the Pakistan players) cannot be ranked.
team_names = pd.read_csv(os.path.join("..", "teams.csv")).drop_duplicates('team_id').set_index('team_id')['team_name']
columns = ['name', 'team', 'balls', 'high_balls', 'pressure_matches', 'ipl_high_balls', 'ipl_pressure_matches',
           'first_innings_share', 'raw_level', 'raw_high', 'level', 'clutch', 'score', 'beat_average']

ranked = {}
for label, table, min_high, source, player in [
    ('batters', batters, 100, balls[balls['legal']], 'striker_id'),
    ('bowlers', bowlers, 120, balls, 'bowler_id'),
]:
    ipl_high = source[(source['comp_group'] == 'IPL') & source['high']].groupby([player, 'match_id']).size()
    table['ipl_high_balls'] = ipl_high.groupby(level=0).sum()
    table['ipl_pressure_matches'] = (ipl_high >= 6).groupby(level=0).sum()
    table[['ipl_high_balls', 'ipl_pressure_matches']] = table[['ipl_high_balls', 'ipl_pressure_matches']].fillna(0)

    overall = (table['high_balls'] >= min_high) & (table['pressure_matches'] >= 10)
    in_ipl = (table['ipl_high_balls'] >= min_high / 2) & (table['ipl_pressure_matches'] >= 5)
    eligible = table[overall & in_ipl].copy()
    eligible['team'] = eligible['team'].map(team_names)
    eligible = eligible.sort_values('score', ascending=False)
    eligible.insert(0, 'rank', np.arange(1, len(eligible) + 1))
    eligible[['rank'] + columns].round(4).to_csv(os.path.join(output_folder, f"04_{label}.csv"))
    ranked[label] = eligible
    left_out = table[overall & ~in_ipl].sort_values('score', ascending=False)
    print(f"{label}: {len(eligible)} eligible, {len(left_out)} left out by the IPL rule. Best of those: {left_out['name'].head(5).tolist()}")

pd.set_option('display.width', 250)
print(ranked['batters'][['rank'] + columns].head(10).round(3).to_string(index=False))

batters: 57 eligible, 10 left out by the IPL rule. Best of those: ['Reeza Hendricks', 'Glenn Phillips', 'Mohammad Rizwan', 'Daryl Mitchell', 'Sikandar Raza']
bowlers: 62 eligible, 23 left out by the IPL rule. Best of those: ['Blessing Muzarabani', 'Haris Rauf', 'Mohammad Nawaz', 'Abrar Ahmed', 'Shaheen Afridi']
 rank             name                        team  balls  high_balls  pressure_matches  ipl_high_balls  ipl_pressure_matches  first_innings_share  raw_level  raw_high  level  clutch  score  beat_average
    1  Abhishek Sharma         Sunrisers Hyderabad   2239       429.0              28.0           284.0                  16.0                0.546      0.392     0.396  0.307     0.0  0.307         0.643
    2 Suryakumar Yadav                       India   2746       493.0              30.0           273.0                  15.0                0.507      0.301     0.278  0.248     0.0  0.248         0.700
    3        Tim David                   Australia   1210       293.0      

In [16]:
print(ranked['bowlers'][['rank'] + columns].head(10).round(3).to_string(index=False))

 rank                name                        team  balls  high_balls  pressure_matches  ipl_high_balls  ipl_pressure_matches  first_innings_share  raw_level  raw_high  level  clutch  score  beat_average
    1      Jasprit Bumrah              Mumbai Indians   1780       374.0              25.0           304.0                  22.0                0.541      0.262     0.018  0.154     0.0  0.154         0.520
    2      Josh Hazlewood Royal Challengers Bengaluru   1515       324.0              22.0           227.0                  17.0                0.487      0.229     0.521  0.127     0.0  0.127         0.773
    3 Matheesha Pathirana         Chennai Super Kings   1279       275.0              17.0           190.0                  12.0                0.457      0.235     0.300  0.121     0.0  0.121         0.824
    4        Nathan Ellis                   Australia    959       228.0              13.0           100.0                   5.0                0.604      0.243     0.190  

In [17]:
# Checks on the draft lists
# 1. How far does shrinking move a player? Raw high pressure value against the score, rank correlation
# 2. Is any one team taking many places? (a player in a strong team could start with a small head start)
# 3. Batters who bat first get fewer high pressure balls, is the top 10 tilted to chasers?
for label in ['batters', 'bowlers']:
    table = ranked[label]
    top = table.head(10)
    print(f"{label}: rank correlation of score with raw high pressure value {table['score'].corr(table['raw_high'], method='spearman'):.2f}, "
          f"with raw level {table['score'].corr(table['raw_level'], method='spearman'):.2f}")
    print(f"   teams in the top 10: {top['team'].value_counts().to_dict()}")
    print(f"   share of balls in the first innings: top 10 {top['first_innings_share'].mean():.2f}, all eligible {table['first_innings_share'].mean():.2f}")

batters: rank correlation of score with raw high pressure value 0.70, with raw level 1.00
   teams in the top 10: {'Sunrisers Hyderabad': 3, 'India': 1, 'Australia': 1, 'England': 1, 'Royal Challengers Bengaluru': 1, 'Gujarat Titans': 1, 'Rajasthan Royals': 1, 'Lucknow Super Giants': 1}
   share of balls in the first innings: top 10 0.55, all eligible 0.52
bowlers: rank correlation of score with raw high pressure value 0.73, with raw level 0.99
   teams in the top 10: {'Kolkata Knight Riders': 2, 'Mumbai Indians': 1, 'Royal Challengers Bengaluru': 1, 'Chennai Super Kings': 1, 'Australia': 1, 'Sunrisers Hyderabad': 1, 'Lucknow Super Giants': 1, 'Bangladesh': 1, 'Gujarat Titans': 1}
   share of balls in the first innings: top 10 0.52, all eligible 0.51


### What step 4 found

- **An average player does not change under pressure.** Value stays within 0.03 runs per ball of 0 in every pressure group, so the step 3 baseline is fair at all pressure levels.
- **No clutch spread above the noise, for batters or bowlers.** With shuffled high pressure labels (no clutch possible) the method still finds a spread of about 0.05 on average, and 90% of shuffles are below 0.10. The real numbers are 0.000 for batters and 0.057 for bowlers. So clutch is set to 0 and both lists are ranked on level.
- **What I cannot claim.** With a made up clutch of 0.10 added, the method finds it for bowlers (0.104) and only half of it for batters (0.053). A clutch skill smaller than about 0.10 runs per ball could exist and this data could not show it.
- **IPL rule.** The report is for the Rajasthan Royals and the IPL 2027 auction, so only players with a real IPL sample are ranked. It leaves out 10 batters and 23 bowlers.
- Draft top 10s are in the cells above and in `outputs/04_batters.csv` and `outputs/04_bowlers.csv`. Step 5 tests if they hold.
- Surprising name : Rajat Patidar is 6th on level (0.295) but only 0.069 on his 132 high pressure balls and above average in 3 of his 10 pressure matches. With clutch at 0, those 132 balls are treated as luck. Step 5's split-half test decides if that is right.